Notebook to plot all figures used in the Supplementary Video V1.

- Source: GLORYS12V1
- Period: February 4th - March 27th

In [30]:
import warnings
warnings.filterwarnings('ignore')

In [1]:
import itertools
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cmocean.cm as cmo
import cartopy.crs as ccrs
import pandas as pd
import gsw
from glob import glob
import matplotlib.patches as patches

import matplotlib.ticker as tkr
from matplotlib.lines import Line2D
from tqdm import tqdm
from datetime import datetime

from xgcm import Grid
from xgcm.autogenerate import generate_grid_ds
import gsw

xr.set_options(keep_attrs=True)

from utils import make_map, subplots_structure, get_shelf_edge

In [2]:
def prepare_isopycnal_surfaces(ds, target):
    ds['sigma_0'] = gsw.sigma0(ds['so'], ds['thetao'])
    
    # Generate 'full' dataset, which includes additional coordinate `depth_left` and appropriate attributes.
    ds_full = generate_grid_ds(ds, {'Z':'depth'})
    
    # Create grid object
    grid = Grid(ds_full, periodic=False)
    
    so_sigma = grid.transform(
      ds_full.so,
      'Z',
      target,
      target_data=ds_full.sigma_0
    )
    
    ds_sigma = xr.Dataset({'so': so_sigma})
    
    return ds_sigma

In [3]:
ref_depth = 50
target = np.arange(24., 25.6, 0.5) # from Silva et al (2023)
zoomed_extent= [-48, -34, -30, -16] # [-45, -39, -27, -22]

datadir = '../data/glorys'

flist = sorted(glob(f"{datadir}/smax_*.nc"))
len(flist)

5

In [4]:
bathy = xr.open_dataset('../data/gebco_updated.nc')
# rename variables for a padronization
bathy = bathy.rename({'elevation': 'depth'})
bathy = bathy.where(bathy['depth'] < 0, drop=True)

# regional selection (for both dataset loaded)
lon_lower, lon_upper = zoomed_extent[0], zoomed_extent[1] #-50, -40
lat_lower, lat_upper = zoomed_extent[2], zoomed_extent[3] #-29, -23

# bathymetry
bathy = bathy.sel(lon=slice(lon_lower, lon_upper),
                lat=slice(lat_lower, lat_upper))

In [7]:
dsall = xr.open_mfdataset(flist)

# cutting a very specific period after an analysis using the entire period
period = ['2022-02-04', '2022-03-27']
dsall = dsall.sel(time=slice(period[0], period[1]))

dsall.dims

Frozen({'depth': 23, 'latitude': 229, 'longitude': 265, 'time': 52})

In [6]:
# load the selected profiles for reference
import sys
sys.path.insert(0, '/home/danilo/phd/thesis/chapter_colab/scripts/')
from load_profiles import load_vertical_profiles

dict_casts = load_vertical_profiles()

Processing cast1...
Processing cast2...
Processing cast3.2...
Processing cast6...
Processing cast7...


In [8]:
# arrows to point features of interest
dict_arrow = {
    '20220204': [(-40.2, -25),(-38.1, -24.5), 'Smax Arrival'],
    '20220205': [(-40.2, -25),(-38.1, -24.5), 'Smax Arrival'],
    '20220206': [(-40.2, -25),(-38.1, -24.5), 'Smax Arrival'],
    '20220207': [(-40.2, -25),(-38.1, -24.5), 'Smax Arrival'],
    '20220208': [(-40.2, -25),(-38.1, -24.5), 'Smax Arrival'],
    '20220209': [(-40.2, -25),(-38.1, -24.5), 'Smax Arrival'],
    '20220210': [(-40.5, -25.4),(-38.3, -25.2), 'Smax near the\n1st observation'],
    '20220211': [(-40.5, -25.4),(-38.3, -25.2), 'Smax near the\n1st observation'],
    '20220212': [(-40.5, -25.4),(-38.3, -25.2), 'Smax near the\n1st observation'],
    '20220213': [(-40.5, -25.4),(-38.3, -25.2), 'Smax near the\n1st observation'],
    '20220214': [(-40.5, -25.4),(-38.3, -25.2), 'Smax near the\n1st observation'],
    '20220215': [(-40.5, -25.4),(-38.3, -25.2), 'Smax near the\n1st observation'],
    '20220216': [(-40.5, -25.4),(-38.3, -25.2), 'Smax near the\n1st observation'],
    '20220217': [(-40.5, -25.4),(-38.3, -25.2), 'Smax near the\n1st observation'],
    '20220218': [(-42., -26.8),(-40.8, -28.2), 'Smax moves\nsouthwards'],
    '20220219': [(-42., -26.8),(-40.8, -28.2), 'Smax moves\nsouthwards'],
    '20220220': [(-42., -26.8),(-40.8, -28.2), 'Smax moves\nsouthwards'],
    '20220221': [(-42., -26.8),(-40.8, -28.2), 'Smax moves\nsouthwards'],
    '20220222': [(-42., -26.8),(-40.8, -28.2), 'Smax moves\nsouthwards'],
    '20220223': [(-42., -26.8),(-40.8, -28.2), 'Smax moves\nsouthwards'],
    '20220224': [(-42., -26.8),(-40.8, -28.2), 'Smax moves\nsouthwards'],
    '20220225': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220226': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220227': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220228': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220301': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220302': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220303': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220304': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220305': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220306': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220307': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220308': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220309': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220310': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220311': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220312': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220313': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220314': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220315': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220316': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220317': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220318': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220319': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220320': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220321': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220322': [(-43., -26.),(-45, -26.2), 'Smax moves\nshoreward'],
    '20220323': [(-43, -24.1),(-43.3, -22.8), 'Smax reaches\nthe shelf edge'],
    '20220324': [(-43, -24.1),(-43.3, -22.8), 'Smax reaches\nthe shelf edge'],
    '20220325': [(-43, -24.1),(-43.3, -22.8), 'Smax reaches\nthe shelf edge'],
    '20220326': [(-43, -24.1),(-43.3, -22.8), 'Smax reaches\nthe shelf edge'],
    '20220327': [(-43, -24.1),(-43.3, -22.8), 'Smax reaches\nthe shelf edge'],
}

In [ ]:
# only to  make easier to plot everything or just a single date for tests
plotHowMany = None # use 1 as test, to plot a single map in this notebook

vmin,vmax = 35,38

# plotting the first observation, at Feb 10th (cast1)
values = dict_casts['cast1']
xlon = values['xlon']
ylat = values['ylat']

for _,itime in tqdm(enumerate(range(dsall.dims['time'])[:plotHowMany])):
    ds = dsall.isel(time=itime)
    datestr = pd.to_datetime(ds.time.values).strftime('%Y-%m-%d')
    
    fig,axes = subplots_structure(2, 1, (18,12), 2)
    
    ax = axes[0]
    ax = make_map(ax=ax, facecolor='lightgray', deltax=2, deltay=1)
    ax.set_extent(zoomed_extent)

    # ax.plot(shelf['lon'], shelf['lat'], 'k--', lw=.5)
    cr = bathy.depth.plot.contour(ax=ax,
                             levels=[-200, -1000, -1500, -2000, -2500, -3000, -4000, -5000], 
                             colors=('k'), linewidths=(0.5), linestyles=('--'))

    idz = np.argmin(np.abs(ds.depth.values - ref_depth))
    
    pc = ds.so.isel(depth=idz).squeeze().plot.pcolormesh(ax=ax, cmap='turbo', vmin=vmin, vmax=vmax, add_colorbar=False)
    # cs = ds.so.isel(depth=idz).squeeze().plot.contour(ax=ax, levels=levels, colors=colors, linestyles=styles)

    for cast,values in dict_casts.items():
        prof = values['prof'] #keys[c]
        xlon = prof.attrs['lon']
        ylat = prof.attrs['lat']
        ax.scatter(xlon, ylat, c='k', s=100)
    
    ds_sigma = prepare_isopycnal_surfaces(ds, target)
    
    dsplot = ds_sigma.mean(dim='sigma_0')
    
    ax = axes[1]
    ax = make_map(ax=ax, facecolor='lightgray', deltax=2, deltay=1)
    ax.set_extent(zoomed_extent)

    # ax.plot(shelf['lon'], shelf['lat'], 'k--', lw=.5)
    cr = bathy.depth.plot.contour(ax=ax,
                             levels=[-200, -1000, -1500, -2000, -2500, -3000, -4000, -5000], 
                             colors=('k'), linewidths=(0.5), linestyles=('--'))
    
    pc = dsplot.so.plot.pcolormesh(ax=ax, x='longitude', y='latitude', cmap='turbo', 
                                           vmin=vmin, vmax=vmax, add_colorbar=False)

    # plotting selected profiles locations
    for cast,values in dict_casts.items():
        prof = values['prof'] #keys[c]
        xlon = prof.attrs['lon']
        ylat = prof.attrs['lat']
        ax.scatter(xlon, ylat, c='k', s=100)
    
    plt.tight_layout()
    
    cax = fig.add_axes([0.05, 0.78, 0.2, 0.02])
    cbar = fig.colorbar(pc, cax=cax, orientation='horizontal')#, format=tkr.FormatStrFormatter('%d'))
    cbar.ax.tick_params(axis='x', labelsize=14)
    cbar.set_label('Salinity [psu]', fontsize=16)
    _ = [ax.set_title('') for ax in axes]
    _ = axes[0].set_title(f'[a] 50-m depth layer: {datestr}', loc='left', fontsize=16, y=0.95)
    _ = axes[1].set_title(r'[b] 25.5 $kg\ m^{-3}$ isopycnal layer', loc='left', fontsize=16, y=0.95)

    # draw rectangle over area of interest
    axes[0].add_patch(patches.Rectangle((-44, -27), 4, 4, linewidth=1, edgecolor='k', ls='--', facecolor='none', zorder=20))
    axes[1].add_patch(patches.Rectangle((-44, -27), 4, 4, linewidth=1, edgecolor='k', ls='--', facecolor='none', zorder=20))

    # checking squares to be plotted based on the date
    if pd.to_datetime(ds.time.values).strftime('%Y%m%d') in dict_arrow.keys():
        loc = dict_arrow[pd.to_datetime(ds.time.values).strftime('%Y%m%d')]
    
        t = axes[1].annotate(loc[2],xy=loc[0], xytext=loc[1],
                 arrowprops={"width": 10, "headwidth": 20, "headlength": 20},
                 horizontalalignment="center", fontsize=14)
        t.set_bbox(dict(facecolor='white', alpha=0.6, edgecolor='white'))


    if plotHowMany != 1:
        plt.savefig(f"../figures/supplementary_movie/glorys_combined_50m_and_isopycnal_{pd.to_datetime(ds.time.values).strftime('%Y%m%d')}.png",
                   dpi=150,
                   bbox_inches='tight')
        plt.close('all')

52it [01:37,  1.88s/it]
